# MNIST Digit Recognition with NumPy

Oldschool implementation of Deep Learning models. Notation and implementation hugely based on Andrew Ng's Coursera course, [Neural Networks and Deep Learning](https://www.coursera.org/learn/neural-networks-deep-learning) - DeepLearning.AI

In [1]:
import random

import numpy as np
#from sklearn.metrics import ConfusionMatrixDisplay

from tensorflow.keras.datasets import mnist

import utils

In [2]:
import yaml

with open("params.yaml") as f:
    params = yaml.safe_load(f)['mnist']

# frequently used params
width = params['img_width']
height = params['img_height']

utils.print_params(params)

img_width: 28
img_height: 28
random_state: 42
batch_size: 128
learning_rate: 0.001
epochs: 5
ly_fc1_out: 512


In [3]:
random.seed(params['random_state'])

In [4]:
from typing import Any

def relu(x) -> np.array:

    return np.maximum(0, x)


def softmax(x, axis=None):
    """Compute the softmax of each element along a specified axis in x.

    Parameters
    ----------
    x: ND-Array
        Input array. Should be of float type for best practice.
    axis: int, optional
        The axis along which the softmax will be computed.
        If None (default), softmax is computed over all elements of the input array.

    Returns
    -------
    ndarray
        An array of the same shape as x with probabilities that sum to 1 along the specified axis.
    """
    # Subtract the maximum value for numerical stability
    x_max = np.max(x, axis=axis, keepdims=True)
    e_x = np.exp(x - x_max)
    # Divide by the sum along the specified axis
    return e_x / np.sum(e_x, axis=axis, keepdims=True)


class layer_fc():
    def __init__(self, n_in, n_out, af=None) -> None:
        #n_in: j (features)
        #n_out: k
        #af: activation function

        # self.w = np.random.random((n_in, n_out))
        # self.b = np.random.random((n_out))
        self.w = (np.random.random((n_in, n_out)) * 2.) - 1.
        self.b = (np.random.random((n_out)) * 2.) - 1.
        self.af = af

        return

    def __call__(self, x: np.array) -> np.array:

        return self.af(np.dot(x, self.w) + self.b)

In [5]:
class MNISTNet():
    def __init__(self):
        self.fc1 = layer_fc(width * height, params['ly_fc1_out'])  # fully connected
        self.fc2 = layer_fc(params['ly_fc1_out'], 10)


        self.fc1 = nn.Linear(width * height, params['ly_fc1_out'])  # fully connected
        self.fc2 = nn.Linear(params['ly_fc1_out'], 10)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.softmax(self.fc2(x), dim=1)

        return x


__Forward pass__

For a feedforward model,

\begin{equation} %\label{eq1}
\hat{y} = f(X) = f^{[L]}(\dots(f^{[2]}(f^{[1]}(X))))
\end{equation}

where $X$ is the input with shape ($B, \dots$), $y$ is the output, and $f$ is the model. Renaming $a^{[0]}\equiv X$ and $a^{[L]}\equiv y$ where $L$ is the number of layers, we can rewrite eq (1) as,

\begin{equation*}
y = f(X)
\end{equation*}

\begin{equation*}
a^{[1]} = f^{[1]} (a^{[0]}),
\end{equation*}

\begin{equation*}
a^{[2]} = f^{[2]} (a^{[1]}),
\end{equation*}

\begin{equation*}
\dots
\end{equation*}

\begin{equation*}
a^{[L]} = f^{[L]} (a^{[L-1]}).
\end{equation*}

For layer $l$,

\begin{equation*} %\label{eq2}
a^{[l]} = f^{[l]} (a^{[l-1]}) = g^{[l]} (z^{[l]}) = g^{[l]} (w^{[l]} a^{[l-1]} + b^{[l]})
\end{equation*}

where,

$a$: output or activation

$g$: activation function

$z$: "cache"

$w$: weights

$b$: bias

$n$: number of units

$m$: batch length

For $X$ with shape $i,j$ where $i$ is the number of instances (batch size) and $j$ is the number of features:


$z_{i}^{k} = w_{j}^{k} X_{i}^{j} + b_{i}^{k}$

Shapes:

$$a^{[L]}: [n^{[L]}, n^{[L-1]}]$$
$$W^{[L]}: [n^{[L]}, n^{[L-1]}]$$
$$W^{[L]}: [n^{[L]}, n^{[L-1]}]$$
$$W^{[L]}: [n^{[L]}, n^{[L-1]}]$$
$$W^{[L]}: [n^{[L]}, n^{[L-1]}]$$
$$W^{[L]}: [n^{[L]}, n^{[L-1]}]$$


___
___
___
__Summary__

\begin{equation*} %\label{eq:activation}
a^{[l]} = g^{[l]} (z^{[l]} (a^{[l-1]}; w^{[l]}, b^{[l]}))
\end{equation*}

where,

\begin{equation*} %\label{eq:activation}
z^{[l]} = (w^{[l]} a^{[l-1]} + b^{[l]})
\end{equation*}

therefore,

\begin{equation*}
\frac{\partial z^{[l]}}{\partial w^{[l]}} = (a^{[l-1]})^T
\qquad
\frac{\partial z^{[l]}}{\partial b^{[l]}} = \mathbf{1}
\end{equation*}

Definitions (to be used later):

\begin{equation*}
dw^{[L]} \equiv \nabla_{w^{[L]}} \mathcal{L},
\qquad
da^{[L]} \equiv \frac{\partial \mathcal{L}}{\partial a^{[L]}},
\qquad
g'^{[L]} \equiv \frac{\partial g^{[L]}}{\partial z^{[L]}}.
\end{equation*}

__Gradient__

\begin{equation*}
\nabla_{w^{[L]}} \mathcal{L} = \frac{\partial \mathcal{L}}{\partial w^{[L]}}
\end{equation*}

\begin{equation*}
\qquad = \frac{\partial \mathcal{L}}{\partial a^{[L]}} \frac{\partial a^{[L]}}{\partial w^{[L]}}
\end{equation*}

\begin{equation*}
\qquad = \frac{\partial \mathcal{L}}{\partial a^{[L]}} 
\frac{\partial g^{[L]}}{\partial z^{[L]}} 
\frac{\partial z^{[L]}}{\partial w^{[L]}}
\end{equation*}

which can be written as,



$$dw^{[L]} = da^{[L]} g'^{[L]} (a^{[l-1]})^T$$

Similarly,

$$db^{[L]} = da^{[L]} g'^{[L]}





___
___
___


__Binary Cross-Entropy__

\begin{equation*}
\mathcal{L} (y, a) = - y\log{a} + (1-y)\log{(1-a)}
\end{equation*}

\begin{equation*}
\frac{\partial\mathcal{L}}{\partial a} = - \frac{y}{a} + \frac{1-y}{1-a}
\end{equation*}

___

\begin{equation}
\frac{\partial f}{\partial b} = \frac{\partial f^{[L]}}{\partial a^{[L]}} \frac{\partial a^{[L]}}{\partial a^{[L-1]}} \dots \frac{\partial f^{[2]}}{\partial a^{[1]}} \frac{\partial f^{[1]}}{\partial a^{[1]}} 
\end{equation}


\begin{equation}
\nabla_{w^{[L]}} \mathcal{L} = \frac{\partial \mathcal{L}}{\partial w^{[L]}} = \frac{\partial f^{[L]}}{\partial a^{[L]}} \frac{\partial a^{[L]}}{\partial a^{[L-1]}} \dots \frac{\partial f^{[2]}}{\partial a^{[1]}} \frac{\partial f^{[1]}}{\partial a^{[1]}} 
\end{equation}



In [6]:
x1 = np.array([1, 2, 3, 4, 5])
x2 = np.array([5, 4, 3, 2, 1])

In [7]:
x1.shape, x2.shape

((5,), (5,))

In [8]:
x1 / x2

array([0.2, 0.5, 1. , 2. , 5. ])

In [9]:
i = 10  # batches
j = 5  # features
k = 3  # output dim

x = np.random.random((i, j))
w = np.random.random((j, k))
b = np.random.random((k))

In [13]:
i = 10  # batch size
j = 5  # features
k = 3  # output dim

l = layer_fc(j, k, relu)
a0 = np.random.random((i, j))

In [16]:
l.w.shape

(5, 3)

In [15]:
a0.shape

(10, 5)

In [ ]:
#l(np.random.random((6, 2)))
#l.w
np.dot(a0, l.w) + l.b

In [ ]:
l(a0)

In [ ]:
l(a0).max()

In [ ]:
#relu(l(a0))

#np.max(np.array([0.]), l(a0))
np.maximum(0.9, l(a0))

In [ ]:
np.random.random?

In [ ]:
# \begin{eqnarray}
    # a^{[1]} &=& 10 \\
    # \dots &=& \\
    # x     &=& 10 - y
# \end{eqnarray}




In [ ]:
## reproducibility
#keras.utils.set_random_seed(params['random_state'])
#experimental.enable_op_determinism()

## Dataset

### Load

In [ ]:
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

In [ ]:
print(train_images.shape, train_labels.shape, test_images.shape, test_labels.shape)
print(set([type(train_images), type(train_labels), type(test_images), type(test_labels)]))
print(set([train_images.dtype, train_labels.dtype, test_images.dtype, test_labels.dtype]))
print(train_images[0].min(), train_images[0].max())

In [ ]:
train_images = train_images\
    .reshape((60000, width * height))\
    .astype("float32") / 255
test_images = test_images\
    .reshape((10000, width * height))\
    .astype("float32") / 255

In [ ]:
print(train_images.shape, train_labels.shape, test_images.shape, test_labels.shape)
print(set([type(train_images), type(train_labels), type(test_images), type(test_labels)]))
print("images.dtype:", set([train_images.dtype, test_images.dtype]))
print("labels.dtype:", set([train_labels.dtype, test_labels.dtype]))
print(train_images[0].min(), train_images[0].max())

### Prepare

In [ ]:
sorted(set(test_labels))

## Model

### Build (architecture)

In [ ]:
## OLD VERSION (works too)
#from keras import models
#from keras import layers
#
#model = models.Sequential()
#model.add(layers.Dense(params['ly_fc1_out'], activation='relu', input_shape=(width * height,)))
#model.add(layers.Dense(10, activation='softmax'))

In [ ]:
model = keras.Sequential([
    layers.Dense(params['ly_fc1_out'], activation="relu"),
    layers.Dense(10, activation="softmax")
])

### Compile

In [ ]:
optimizer = optimizers.RMSprop(learning_rate=params['learning_rate'])

In [ ]:
model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

TODO: WARNING:tensorflow:TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.

### Training loop (fit)

In [ ]:
model.fit(train_images, train_labels, epochs=params['epochs'], batch_size=params['batch_size'])

## Results

### Predictions

In [ ]:
test_digits = test_images[0:10]
predictions = model.predict(test_digits)
predictions[0]

In [ ]:
predictions[0].argmax()

### Evaluation (accuracy)

In [ ]:
test_loss, test_acc = model.evaluate(test_images, test_labels)
print(f"test_acc: {test_acc:.2%}")

In [ ]:
pred = np.argmax(model.predict(test_images), axis=1)

In [ ]:
## OLD VERSION
#real = np.argmax(test_labels, axis=1)

real = test_labels.astype(np.int64)

In [ ]:
notequals = (pred != real)
notequals = np.where(notequals)[0]
len(notequals)

In [ ]:
# check accuracy
print(f"{(len(pred) - len(notequals)) / len(pred):.2%}")

### Show

In [ ]:
## OLD VERSION (fix needed for some reason)
#import os
#os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

True label + Image:

In [ ]:
idx = 4
digit = train_images[idx]
digit = digit.reshape(width, height)
print(train_labels[idx])
utils.imshow(digit)

Wrongly labeled digits in test:

In [ ]:
foo = utils.show_wrong_preds(test_images, real, pred, notequals, width, height)

Confusion matrix:

In [ ]:
foo = ConfusionMatrixDisplay.from_predictions(real, pred, labels=range(10), cmap='nipy_spectral_r')

Specific mistakes:

In [ ]:
# 2 != 0
for i in range(len(real)):
    if real[i] == 2 and pred[i] == 0:
        digit = test_images[i]
        digit = digit.reshape(width, height)
        type(digit), digit.shape
        utils.imshow(digit)